# Regularizar todos los umbrales

`RIconduitex5_5.py` no se toca. Este notebook es el laboratorio: cada rampa contra el `if` original, con el **epsilon escogido** (línea gruesa) y uno más flaco / más gordo para ver si se pega de más o de menos.

`EPS` de acá es solo para **mirar** las rampas. El solver no lo lee salvo que se lo pases a `RIconduitex5_5_suave_f(..., eps_frag=EPS_RUN['frag'], ...)`.

Si el perfil suave “no se ve como antes” (ξ 0.25→0.35, φ salta a −20 km): no es el `EPS`. Era un `softplus` extra sobre `dx/dz`. `softplus(0, eps) ≈ 0.69·eps` por metro y los cristales se inflan. Ahora `tasa_xi` solo suaviza `f2` y `f3`, igual que el `max(0,·)` del original.


In [ ]:
from umbrales_reg import *
import numpy as np
import matplotlib.pyplot as plt

# ── EPS: anchos para MIRAR la rampa (el tablero de abajo) ─
# Esto NO entra al solver. Para el DAE usá EPS_RUN más abajo.
EPS = dict(
    phi=0.01, frag=0.04, henry=1e-3, re=300.0, xi=1e-3, rb=0.05,
)
# ── EPS_RUN: anchos para CORRER el DAE (pegados al if) ───
# henry=1e-4 todavía es ~100× el test en Pcrit (~1e-6). Dejá 1e-6.
# xi solo suaviza f2 y f3; no se aplica otra vez a dx/dz.
EPS_RUN = dict(
    phi=0.001, frag=0.001, henry=1e-6, re=10.0, xi=1e-4, rb=0.01,
)

PHICRIT = 0.70
LIM1, LIM2 = 0.20, 0.21
WR = 16.0
PATM = 101325.0
CS = (461.11 * 1243.15) ** 0.5


def rampa_vs_switch(ax, x, y_sw, curvas, xlabel, title, xstar=None):
    """curvas = [(y, label, kw), ...]  la primera después del switch es el escogido."""
    ax.plot(x, y_sw, 'k', lw=2.2, label='switch')
    for i, (y, lab, kw) in enumerate(curvas):
        ax.plot(x, y, label=lab, **kw)
    if xstar is not None:
        ax.axvline(xstar, color='gray', lw=1)
    ax.set_xlabel(xlabel, fontweight='bold')
    ax.set_title(title)
    ax.legend()


print('vista', EPS)
print('corrida', EPS_RUN)


## Tablero (todas las variables, epsilon escogido en grueso)
Si alguna rampa se ve demasiado ancha o se pega al switch, tocá ese `EPS` y re-ejecutá.

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(13.5, 10.5))
kw_e = dict(lw=2.4)
kw_f = dict(ls='--', lw=1.4)
kw_g = dict(ls=':', lw=1.6)

phi = np.linspace(0.40, 0.95, 400)
rampa_vs_switch(axs[0,0], phi, switch(phi, PHICRIT), [
    (s_frag(phi, PHICRIT, EPS['frag']), f"escogido {EPS['frag']}", kw_e),
    (s_frag(phi, PHICRIT, 0.01), '0.01', kw_f),
    (s_frag(phi, PHICRIT, 0.10), '0.10', kw_g),
], 'phi', 'fragmentación', PHICRIT)

phiw = np.linspace(0.05, 0.95, 500)
w1, w2, w3, w4 = pesos_regimen(phiw, LIM1, LIM2, PHICRIT, EPS['phi'], EPS['frag'])
axs[0,1].plot(phiw, w1, lw=2, label='w1')
axs[0,1].plot(phiw, w2, lw=2, label='w2')
axs[0,1].plot(phiw, w3, lw=2, label='w3')
axs[0,1].plot(phiw, w4, lw=2, label='w4')
axs[0,1].plot(phiw, w1+w2+w3+w4, 'k--', lw=1, label='suma')
axs[0,1].set_title(f"regímenes  phi={EPS['phi']} frag={EPS['frag']}")
axs[0,1].legend(ncol=3, fontsize=8)

test = np.linspace(-0.01, 0.01, 400)
rampa_vs_switch(axs[0,2], test, switch(test, 0.0), [
    (s_henry(test, EPS['henry']), f"escogido {EPS['henry']}", kw_e),
    (s_henry(test, 1e-4), '1e-4', kw_f),
    (s_henry(test, 3e-3), '3e-3', kw_g),
], 'test', 'Henry', 0.0)

Re = np.linspace(0, 5000, 400)
rampa_vs_switch(axs[1,0], Re, switch(Re, RE_CRIT), [
    (s_re(Re, RE_CRIT, EPS['re']), f"escogido {EPS['re']}", kw_e),
    (s_re(Re, RE_CRIT, 100), '100', kw_f),
    (s_re(Re, RE_CRIT, 600), '600', kw_g),
], 'Re', 'Reynolds', RE_CRIT)

x = np.linspace(-0.02, 0.02, 400)
rampa_vs_switch(axs[1,1], x, np.maximum(0, x), [
    (softplus(x, EPS['xi']), f"escogido {EPS['xi']}", kw_e),
    (softplus(x, 3e-4), '3e-4', kw_f),
    (softplus(x, 4e-3), '4e-3', kw_g),
], 'x', 'cristales')

rb = np.linspace(0, WR, 400)
rampa_vs_switch(axs[1,2], rb/WR, (rb < 0.5*WR).astype(float), [
    (g_rb(rb, WR, EPS['rb']), f"escogido {EPS['rb']}", kw_e),
    (g_rb(rb, WR, 0.02), '0.02', kw_f),
    (g_rb(rb, WR, 0.12), '0.12', kw_g),
], 'rb / R', 'coalescencia g_r', 0.5)

z = np.linspace(-20, 2, 300)
axs[2,0].plot(z, residual_explosivo(PATM, PATM, CS, CS, 0.8, PHICRIT, z)['r_z'], lw=2)
axs[2,0].axhline(0, color='k', lw=1); axs[2,0].axvline(-5, color='gray')
axs[2,0].set_title('tiro ex  r_z'); axs[2,0].set_xlabel('z_exit')

zef = np.linspace(-8, 2, 300)
axs[2,1].plot(zef, residual_efusivo(PATM, PATM, 0.4, PHICRIT, zef)['r_z'], lw=2)
axs[2,1].axvspan(-2, 0, color='0.85'); axs[2,1].axhline(0, color='k', lw=1)
axs[2,1].set_title('tiro ef  r_z'); axs[2,1].set_xlabel('z_exit')

rP = np.linspace(-2, 2, 300)
rampa_vs_switch(axs[2,2], rP, np.minimum(rP**2, 1.0), [
    (softmin2(rP**2, 1.0, 0.2), 'escogido 0.2', kw_e),
    (softmin2(rP**2, 1.0, 0.05), '0.05', kw_f),
    (softmin2(rP**2, 1.0, 0.6), '0.6', kw_g),
], 'r_P', 'P_atm O choque')

fig.suptitle('todas las rampas  (línea gruesa = epsilon escogido)', fontweight='bold')
plt.tight_layout(); plt.show()

phi_a = np.linspace(0.4, 0.95, 2000)
print('ancho 10-90 fragmentación (escogido):',
      ancho_10_90(phi_a, s_frag(phi_a, PHICRIT, EPS['frag'])))
print('ancho 10-90 Henry (escogido):',
      ancho_10_90(np.linspace(-0.01, 0.01, 2000), s_henry(np.linspace(-0.01, 0.01, 2000), EPS['henry'])))
print('ancho 10-90 Re (escogido):',
      ancho_10_90(np.linspace(0, 5000, 2000), s_re(np.linspace(0, 5000, 2000), RE_CRIT, EPS['re'])))


## 1. Fragmentación $\phi_{\mathrm{crit}}$

In [ ]:
phi = np.linspace(0.40, 0.95, 400)
fig, ax = plt.subplots(figsize=(7, 4))
rampa_vs_switch(ax, phi, switch(phi, PHICRIT), [
    (s_frag(phi, PHICRIT, EPS['frag']), f"escogido {EPS['frag']}", dict(lw=2.4)),
    (s_frag(phi, PHICRIT, 0.01), '0.01', dict(ls='--', lw=1.4)),
    (s_frag(phi, PHICRIT, 0.10), '0.10', dict(ls=':', lw=1.6)),
], 'phi', 'fragmentación', PHICRIT)
ax.set_ylabel('s_f', fontweight='bold')
plt.tight_layout(); plt.show()
print('ancho 10-90 escogido', ancho_10_90(phi, s_frag(phi, PHICRIT, EPS['frag'])))


## 2. Regímenes $w_1,w_2,w_3,w_4$ (tienen que sumar 1)

In [ ]:
phi = np.linspace(0.05, 0.95, 500)
w1, w2, w3, w4 = pesos_regimen(phi, LIM1, LIM2, PHICRIT, EPS['phi'], EPS['frag'])
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(phi, w1, lw=2, label='w1 Stokes')
ax.plot(phi, w2, lw=2, label='w2 mezcla perm.')
ax.plot(phi, w3, lw=2, label='w3 Darcy/inercial')
ax.plot(phi, w4, lw=2, label='w4 fragmentado')
ax.plot(phi, w1+w2+w3+w4, 'k--', lw=1, label='suma')
for x in (LIM1, LIM2, PHICRIT):
    ax.axvline(x, color='gray', lw=1)
ax.set_xlabel('phi', fontweight='bold')
ax.legend(ncol=2); ax.set_title('regímenes n_eq')
plt.tight_layout(); plt.show()
print('min suma', float((w1+w2+w3+w4).min()), 'max', float((w1+w2+w3+w4).max()))


## 3. Henry (`test`)

In [ ]:
test = np.linspace(-0.01, 0.01, 400)
fig, ax = plt.subplots(figsize=(7, 4))
rampa_vs_switch(ax, test, switch(test, 0.0), [
    (s_henry(test, EPS['henry']), f"escogido {EPS['henry']}", dict(lw=2.4)),
    (s_henry(test, 1e-4), '1e-4', dict(ls='--', lw=1.4)),
    (s_henry(test, 3e-3), '3e-3', dict(ls=':', lw=1.6)),
], 'test (agua exsuelta)', 'Henry', 0.0)
ax.set_ylabel('s_H', fontweight='bold')
plt.tight_layout(); plt.show()
print('ancho 10-90 escogido', ancho_10_90(test, s_henry(test, EPS['henry'])))


## 4. Reynolds

In [ ]:
Re = np.linspace(0, 5000, 400)
fig, ax = plt.subplots(figsize=(7, 4))
rampa_vs_switch(ax, Re, switch(Re, RE_CRIT), [
    (s_re(Re, RE_CRIT, EPS['re']), f"escogido {EPS['re']}", dict(lw=2.4)),
    (s_re(Re, RE_CRIT, 100), '100', dict(ls='--', lw=1.4)),
    (s_re(Re, RE_CRIT, 600), '600', dict(ls=':', lw=1.6)),
], 'Re', 'Re > 2200', RE_CRIT)
plt.tight_layout(); plt.show()
print('ancho 10-90 escogido', ancho_10_90(Re, s_re(Re, RE_CRIT, EPS['re'])))


## 5. Cristales (`max` → softplus)

In [ ]:
x = np.linspace(-0.02, 0.02, 400)
fig, ax = plt.subplots(figsize=(7, 4))
rampa_vs_switch(ax, x, np.maximum(0, x), [
    (softplus(x, EPS['xi']), f"escogido {EPS['xi']}", dict(lw=2.4)),
    (softplus(x, 3e-4), '3e-4', dict(ls='--', lw=1.4)),
    (softplus(x, 4e-3), '4e-3', dict(ls=':', lw=1.6)),
], 'x', 'cristales')
plt.tight_layout(); plt.show()


## 6. Coalescencia (rampas que apagan)

In [ ]:
rb = np.linspace(0, WR, 400)
phi = np.linspace(0.2, 0.95, 400)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
rampa_vs_switch(axs[0], rb/WR, (rb < 0.5*WR).astype(float), [
    (g_rb(rb, WR, EPS['rb']), f"escogido {EPS['rb']}", dict(lw=2.4)),
    (g_rb(rb, WR, 0.02), '0.02', dict(ls='--', lw=1.4)),
    (g_rb(rb, WR, 0.12), '0.12', dict(ls=':', lw=1.6)),
], 'rb / R', 'g_r', 0.5)
rampa_vs_switch(axs[1], phi, (phi < PHICRIT).astype(float), [
    (g_phi(phi, PHICRIT, EPS['frag']), f"escogido {EPS['frag']}", dict(lw=2.4)),
    (g_phi(phi, PHICRIT, 0.01), '0.01', dict(ls='--', lw=1.4)),
    (g_phi(phi, PHICRIT, 0.10), '0.10', dict(ls=':', lw=1.6)),
], 'phi', 'g_phi', PHICRIT)
plt.tight_layout(); plt.show()


## 7. Residual del tiro explosivo (queremos todo en 0)

In [ ]:
P = np.linspace(0.2*PATM, 2.2*PATM, 300)
ug = np.linspace(0.2*CS, 1.8*CS, 300)
phi = np.linspace(0.4, 0.95, 300)
z = np.linspace(-20, 2, 300)
exP = residual_explosivo(P, PATM, CS, CS, 0.8, PHICRIT, 0.0)
exU = residual_explosivo(PATM, PATM, ug, CS, 0.8, PHICRIT, 0.0)
exF = residual_explosivo(PATM, PATM, CS, CS, phi, PHICRIT, 0.0)
exZ = residual_explosivo(PATM, PATM, CS, CS, 0.8, PHICRIT, z)
fig, axs = plt.subplots(2, 2, figsize=(10, 7))
axs[0,0].plot(P/1e5, exP['r_P']/1e5, lw=2); axs[0,0].axhline(0, color='k', lw=1); axs[0,0].set_title('r_P'); axs[0,0].set_xlabel('P (bar)')
axs[0,1].plot(ug/CS, exU['r_c']/CS, lw=2); axs[0,1].axhline(0, color='k', lw=1); axs[0,1].set_title('r_c'); axs[0,1].set_xlabel('ug / cs')
axs[1,0].plot(phi, exF['r_phi'], lw=2); axs[1,0].axhline(0, color='k', lw=1); axs[1,0].axvline(PHICRIT, color='gray'); axs[1,0].set_title('r_phi (0 si fragmentó)'); axs[1,0].set_xlabel('phi')
axs[1,1].plot(z, exZ['r_z'], lw=2); axs[1,1].axhline(0, color='k', lw=1); axs[1,1].axvline(-5, color='gray'); axs[1,1].set_title('r_z (0 si z>=-5)'); axs[1,1].set_xlabel('z_exit')
plt.tight_layout(); plt.show()


## 8. Residual del tiro efusivo

In [ ]:
P = np.linspace(0.2*PATM, 2.2*PATM, 300)
phi = np.linspace(0.4, 0.95, 300)
z = np.linspace(-8, 2, 300)
efP = residual_efusivo(P, PATM, 0.4, PHICRIT, -1.0)
efF = residual_efusivo(PATM, PATM, phi, PHICRIT, -1.0)
efZ = residual_efusivo(PATM, PATM, 0.4, PHICRIT, z)
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8))
axs[0].plot(P/1e5, efP['r_P']/1e5, lw=2); axs[0].axhline(0, color='k', lw=1); axs[0].set_title('r_P')
axs[1].plot(phi, efF['r_phi'], lw=2); axs[1].axhline(0, color='k', lw=1); axs[1].axvline(PHICRIT, color='gray'); axs[1].set_title('r_phi (0 si NO fragmentó)')
axs[2].plot(z, efZ['r_z'], lw=2); axs[2].axhline(0, color='k', lw=1); axs[2].axvspan(-2, 0, color='0.85'); axs[2].set_title('r_z (0 en (-2,0])')
plt.tight_layout(); plt.show()


## 9. Explosivo: $P_{atm}$ **o** choque (`softmin`)

In [ ]:
rP = np.linspace(-2, 2, 300)
rc0 = 1.0
fig, ax = plt.subplots(figsize=(7, 4))
rampa_vs_switch(ax, rP, np.minimum(rP**2, rc0**2), [
    (softmin2(rP**2, rc0**2, 0.2), 'escogido 0.2', dict(lw=2.4)),
    (softmin2(rP**2, rc0**2, 0.05), '0.05', dict(ls='--', lw=1.4)),
    (softmin2(rP**2, rc0**2, 0.6), '0.6', dict(ls=':', lw=1.6)),
], 'r_P  (r_c = 1)', 'P_atm O choque')
plt.tight_layout(); plt.show()


## Solver suave (donde haya IDA)

Mismos datos que `mainconduit5_5`. El dict `EPS` de arriba **no** entra: hay que pasar `EPS_RUN`.

El perfil “como antes” tiene ξ ≈ 0.250–0.252 y φ que sube de a poco hasta fragmentar cerca del cráter. Si ξ se va a 0.35 y φ salta a −20 km, es el leak de `softplus` en `dx/dz` (ya corregido), no el ancho de las rampas.

Si el tiro **no cierra**: no interpolar al umbral exacto (ese y0 no es solución del DAE y n_eq=2/3/4 no arranca). El suave ahora corta como el original (primer punto que ya cruzó) y en n_eq 1–3 usa las mismas fuerzas, sin mezclar fragmentación.

La cámara de Calbuco está en `H = −7000` m. El eje a −70 km del original es el relleno adiabático / un `Hi` mal recortado; el suave arma la columna desde `H` hasta `Hi` (exsolución ~ −3.5 km).


In [ ]:
# Descomentá donde haya IDA (scikits.odes). Mismo unpack que mainconduit5_5.
#
# from RIconduitex5_5_suave import *
# from calbuco2015d import radius1, overP1, h2o1, T1, xi1, geometry
#
# radius = radius1
# Pressure = overP1
# (zsolex, soluex, countex, vex, rho_m, rbubex, viscex, rho_tiex,
#  fragcritex, phicritex, limphi1ex, limphi2ex, coex, xiex, xfinalex, cgex) = \
#     RIconduitex5_5_suave_f(
#         radius, Pressure, h2o1, T1, xi1,
#         eps_frag=EPS_RUN['frag'], eps_phi=EPS_RUN['phi'],
#         eps_henry=EPS_RUN['henry'], eps_re=EPS_RUN['re'],
#         eps_xi=EPS_RUN['xi'], eps_rb=EPS_RUN['rb'])
#
# solex = countex < 49
# wr = radius
# Qex = vex * np.pi * wr ** 2
# print('count', countex, 'ok', solex, 'MER', Qex * rho_tiex)
# print('xi min/max', soluex[:, 3].min(), soluex[:, 3].max())
# print('z min/max', zsolex.min(), zsolex.max())


## Por qué ξ se iba a 0.35 (leak de softplus)

`softplus(0, ε) = ε log 2 ≈ 0.69 ε`. Si se aplica al producto que ya es 0 (columna subsaturada), `dx/dz` vale ~7e-5 /m con `ε=1e-4` y en 15 km los cristales suben ~0.10. El original tiene `max(0, 0) = 0`.


In [ ]:
from umbrales_reg import softplus, tasa_xi

eps = EPS_RUN['xi']
tcar = 2 * 3600
xi0, xmax = 0.25, 0.50
# columna todavía subsaturada: f2_raw < 0 como en el tramo φ=0
f2_raw = -0.01
f2, xteo, f3, dx_ok = tasa_xi(f2_raw, xi0, xi0, xmax, tcar, eps)
dx_leak = float(softplus((xmax - xi0) * softplus(f2_raw, eps)
                         * softplus(1 - xi0 / max(xteo, 1e-30), eps) / tcar, eps))
print('tasa_xi (bien)   dx/dz =', dx_ok)
print('softplus(producto) dx/dz =', dx_leak, '   ≈ 0.69*eps =', 0.69 * eps)
print('metros para llegar a 0.35 con el leak:', (0.35 - xi0) / dx_leak)

L = np.linspace(0, 4000, 400)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(L, np.minimum(xmax, xi0 + dx_ok * L), lw=2.2, label='tasa_xi  (como el original)')
ax.plot(L, np.minimum(xmax, xi0 + dx_leak * L), lw=2.2, label='softplus extra sobre dx/dz')
ax.axhline(0.252, color='gray', lw=1, ls='--', label='como antes (~0.252)')
ax.axhline(0.35, color='0.6', lw=1, ls=':', label='suave viejo (~0.35)')
ax.set_xlabel('metros integrados con tasa filtrada')
ax.set_ylabel('crystal content')
ax.set_ylim(0.24, 0.52)
ax.set_title('el leak, no el EPS, hincha ξ a 0.35')
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# 6 paneles como mainconduit5_5 (después de correr la celda del solver)
# if solex:
#     fig, axs = plt.subplots(1, 6, figsize=(18, 6))
#     axs[0].semilogx(soluex[:, 0], zsolex, linewidth=2)
#     axs[0].set_xlabel('Pressure (Pa)', fontweight='bold', fontsize=14)
#     axs[0].set_ylabel('Depth (m)', fontweight='bold', fontsize=14)
#     axs[1].plot(soluex[:, 1], zsolex, linewidth=2)
#     axs[1].set_xlabel('gas volume fraction', fontweight='bold', fontsize=14)
#     axs[1].set_ylabel('Depth (m)', fontweight='bold', fontsize=14)
#     axs[2].semilogx(soluex[:, 4], zsolex, label='Velocity liq', linewidth=2)
#     axs[2].semilogx(soluex[:, 5], zsolex, '--', label='Velocity gas', linewidth=2)
#     axs[2].set_xlabel('velocity (m/s)', fontweight='bold', fontsize=14)
#     axs[2].set_ylabel('Depth (m)', fontweight='bold', fontsize=14)
#     axs[2].legend()
#     fig.text(0.65, 0.87, f'mass flow rate = {Qex*rho_tiex:.3g} kg/s',
#              bbox=dict(facecolor='white', edgecolor='black'))
#     axs[3].plot(soluex[:, 2], zsolex, linewidth=2)
#     axs[3].set_xlabel('Nd', fontweight='bold', fontsize=14)
#     axs[3].set_ylabel('Depth (m)', fontweight='bold', fontsize=14)
#     axs[4].plot(soluex[:, 3], zsolex, linewidth=2)
#     axs[4].set_xlabel('crystal content', fontweight='bold', fontsize=14)
#     axs[4].set_ylabel('Depth (m)', fontweight='bold', fontsize=14)
#     axs[5].semilogx(viscex, zsolex, linewidth=2)
#     axs[5].set_xlabel('viscosity (Pa.s)', fontweight='bold', fontsize=14)
#     axs[5].set_ylabel('Depth (m)', fontweight='bold', fontsize=14)
#     for ax in axs:
#         ax.set_ylim(min(zsolex.min(), -7000) - 200, 200)
#     plt.tight_layout(); plt.show()


## DAE unificado (un solo IDA)

Campo \(f=\sum w_i f_i\) con `pesos_regimen` (\(w_1+\cdots+w_4=1\)). Mismo estado de 6 variables en todo el conducto; no se corta en `limphi` / `phicrit`. El suave de cuatro tramos sigue igual.

`RIconduitex5_5_unificado.py` es autónomo: no pide `RIconduitex5_5_unificado_f` a suave (el suave de `ambos-casos` no lo tiene). Tiene que estar `umbrales_reg.py` con `tasa_xi` y `pesos_regimen` al lado.

Descomentá donde haya IDA. Compará ξ y φ con el de cuatro tramos: si ε es chico deberían parecerse; el operador acá sí es \(C^\infty\) en \(\varphi_{\mathrm{crit}}\).


In [ ]:
# from RIconduitex5_5_unificado import RIconduitex5_5_unificado_f
# from calbuco2015d import radius1, overP1, h2o1, T1, xi1
#
# (zsol_u, solu_u, count_u, v_u, rho_m_u, rbub_u, visc_u, rho_ti_u, *_) = \
#     RIconduitex5_5_unificado_f(
#         radius1, overP1, h2o1, T1, xi1,
#         eps_frag=EPS_RUN['frag'], eps_phi=EPS_RUN['phi'],
#         eps_henry=EPS_RUN['henry'], eps_re=EPS_RUN['re'],
#         eps_xi=EPS_RUN['xi'], eps_rb=EPS_RUN['rb'])
# print('unificado count', count_u, 'ok', count_u < 49, 'v', v_u)
# print('xi min/max', solu_u[:, 3].min(), solu_u[:, 3].max())
# print('z min/max', zsol_u.min(), zsol_u.max(), 'phi exit', solu_u[-1, 1])
#
# fig, axs = plt.subplots(1, 6, figsize=(18, 6))
# axs[0].semilogx(solu_u[:, 0], zsol_u, linewidth=2)
# axs[0].set_xlabel('Pressure (Pa)'); axs[0].set_ylabel('Depth (m)')
# axs[1].plot(solu_u[:, 1], zsol_u, linewidth=2)
# axs[1].set_xlabel('gas volume fraction')
# axs[2].semilogx(solu_u[:, 4], zsol_u, label='liq', linewidth=2)
# axs[2].semilogx(solu_u[:, 5], zsol_u, '--', label='gas', linewidth=2)
# axs[2].legend(); axs[2].set_xlabel('velocity (m/s)')
# axs[3].plot(solu_u[:, 2], zsol_u, linewidth=2); axs[3].set_xlabel('Nd')
# axs[4].plot(solu_u[:, 3], zsol_u, linewidth=2); axs[4].set_xlabel('crystal content')
# axs[5].semilogx(visc_u, zsol_u, linewidth=2); axs[5].set_xlabel('viscosity (Pa.s)')
# fig.suptitle('unificado  (un IDA, pesos_regimen)')
# plt.tight_layout(); plt.show()
